# Tool Development Sandbox

In [15]:
import math
from dataclasses import dataclass

## Cabling Tools

### Optical Power Budget Calculator

Optical power budget calculator; [issue link](https://github.com/eronlloyd/ictdesigntools/issues/2)

In [218]:
@dataclass(frozen=True)
class Transmitter:
    launch_power_dbm: float  #TODO: should these be decimal?

    def __post_init__(self):
        if self.launch_power_dbm <= -50:
            raise ValueError("Launch power must be greater than -50 dBm.")

@dataclass(frozen=True)
class Receiver:
    receive_sensitivity_dbm: float  #TODO: should these be decimal?

    def __post_init__(self):
        if self.receive_sensitivity_dbm <= -50:
            raise ValueError("Receiver sensitivity must be greater than -50 dBm.")

@dataclass(frozen=True)
class PowerBudget:
    budget: int
    safety_margin: float=2

    def __sub__(self, other: object) -> float:
        # Let the loss budget be subtracted from the power budget
        if isinstance(other, LossBudget):
            return round(self.budget - other.budget, 3)
            
        return NotImplemented

def optical_power_budget(tx, rx):
    budget = tx.launch_power_dbm - rx.receive_sensitivity_dbm
    return PowerBudget(budget)

In [219]:
tx = Transmitter(launch_power_dbm=-5)
rx = Receiver(receive_sensitivity_dbm=-25)

power_budget = optical_power_budget(tx, rx)
print(power_budget)

PowerBudget(budget=20, safety_margin=2)


In [220]:
tx = Transmitter(launch_power_dbm=0)
rx = Receiver(receive_sensitivity_dbm=-25)

power_budget = optical_power_budget(tx, rx)
print(power_budget)

PowerBudget(budget=25, safety_margin=2)


### Optical Loss Budget Calculator

Optical loss budget calculator; [issue link](https://github.com/eronlloyd/ictdesigntools/issues/1)

In [221]:
@dataclass(frozen=True)
class Cable:
    cable_loss_db_km: float
    cable_length_m: float
    cable_type: str='G.652'

    def __post_init__(self):
        if self.cable_length_m <= 0:
            raise ValueError("Cable length must be greater than 0 m.")

    @property
    def loss_db(self) -> float:
        return self.cable_loss_db_km * self.cable_length_m

@dataclass(frozen=True)
class Splice:
    splice_type: str='fusion'
    loss_db: float=0.1

    def __post_init__(self):
        if self.loss_db <= 0:
            raise ValueError("Splice loss must be greater than 0 dB.")

@dataclass(frozen=True)
class Connector:
    connector_type: str='LC/UPC'
    loss_db: float=0.5

    def __post_init__(self):
        if self.loss_db <= 0:
            raise ValueError("Connector loss must be greater than 0 dB.")

@dataclass(frozen=True)
class LossBudget:
    budget: int

def optical_loss_budget(cables, splices, connectors) -> float:
    budget = round(sum(
        [cable.loss_db for cable in cables] +
        [splice.loss_db for splice in splices] +
        [connector.loss_db for connector in connectors]), 3)
    return LossBudget(budget)

In [222]:
cables = [Cable(cable_length_m=40, cable_loss_db_km=0.37)]
splices = [Splice() for splice in range(21)]
connectors = [Connector() for connector in range(2)]

loss_budget = optical_loss_budget(cables, splices, connectors)
loss_budget

LossBudget(budget=17.9)

### Net Loss Calculator

In [223]:
def optical_loss_calculator(power_budget, loss_budget):
    print(f'Power budget is {round(power_budget.budget, 3)}dB, and loss budget is {round(loss_budget.budget, 3)}dB.')
    loss = power_budget - loss_budget
    margin = loss - power_budget.safety_margin
    print(f'The net budget is {round(loss, 3)}dB and the margin of the safety factor is {round(margin, 3)}dB')
    if loss > power_budget.safety_margin:
        print(f'Budget is {round(margin, 3)}dB over the required {power_budget.safety_margin}dB safety margin.')
        return loss
    else:
        print(f'Budget is {round(margin, 3)}dB under the required {power_budget.safety_margin}dB safety margin.')
        raise Exception

In [234]:
tx = Transmitter(launch_power_dbm=-4.9)
rx = Receiver(receive_sensitivity_dbm=-25)

power_budget = optical_power_budget(tx, rx)
print(power_budget)
optical_loss_calculator(power_budget, loss_budget)

PowerBudget(budget=20.1, safety_margin=2)
Power budget is 20.1dB, and loss budget is 17.9dB.
The net budget is 2.2dB and the margin of the safety factor is 0.2dB
Budget is 0.2dB over the required 2dB safety margin.


2.2

In [225]:
tx = Transmitter(launch_power_dbm=-4)
rx = Receiver(receive_sensitivity_dbm=-25)

power_budget = optical_power_budget(tx, rx)
print(power_budget)
optical_loss_calculator(power_budget, loss_budget)

PowerBudget(budget=21, safety_margin=2)
Power budget is 21dB, and loss budget is 17.9dB.
The net budget is 3.1dB and the margin of the safety factor is 1.1dB
Budget is 1.1dB over the required 2dB safety margin.


3.1